# Evo Aggregated Statistics to MongoDB

This notebook calculates aggregated statistics from Evo downhole objects using the MCP data analysis utilities (shared with the tools) and stores them in a MongoDB collection. 

**Objectives**: 
- Test a basic implementation of the Evo MCP utilities > mongo DB integration
- Experiment with the calculated statistics to determine what is useful
- Assess performance on querying those statistics over a number of objects

**Steps:**
- Connects to Evo platform via hijacked OAuth token 
- Calculates interval statistics (length-weighted mean, accumulation, etc.)
- Calculates per-hole statistics
- Stores results with timestamps in MongoDB for tracking
- Analyzes the performance with and without indexing

**Prerequisites:**
- MongoDB running locally 
- Evo MCP configured with valid credentials in `.env`
- `pymongo` installed

#### Setup

In [2]:
import sys
import pandas as pd
import bson
import json
from pathlib import Path
from datetime import datetime, timezone
from uuid import UUID

# Add src directory to path for imports
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# MongoDB client
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

# Evo MCP utilities
from evo_mcp.context import evo_context, ensure_initialized
from evo_mcp.utils.data_analysis_utils import (
    get_downhole_collection,
    download_interval_data,
    download_downhole_intervals_data,
    calculate_interval_statistics,
    calculate_statistics_by_hole,
    analyze_gaps,
    calculate_multi_grade_statistics,
    get_collection_info,
    get_object_type,
)

#### MongoDB onfiguration

Define the MongoDB connection settings and Evo workspace/object parameters.

In [ ]:
# Load MongoDB connection parameters from config file

config_path = Path.cwd() / "mongo_config.json"
if config_path.exists():
    with open(config_path, 'r') as f:
        mongo_config = json.load(f)
    
    # Check if Atlas credentials are provided
    if "username" in mongo_config and "password" in mongo_config and "cluster_url" in mongo_config:
        protocol = mongo_config.get("protocol", "mongodb+srv")
        username = mongo_config["username"]
        password = mongo_config["password"]
        cluster_url = mongo_config["cluster_url"]
        MONGO_URI = f"{protocol}://{username}:{password}@{cluster_url}"
        print(f"Using MongoDB Atlas ({protocol}): {cluster_url.split('/')[0]}")
    else:
        # Fall back to local MongoDB
        protocol = mongo_config.get("protocol", "mongodb")
        host = mongo_config.get("host", "localhost")
        port = mongo_config.get("port", 27017)
        MONGO_URI = f"{protocol}://{host}:{port}/"
        print(f"Using local MongoDB: {host}:{port}")
    
    MONGO_DB_NAME = mongo_config.get("database", "evo_statistics")
    MONGO_COLLECTION_NAME = mongo_config.get("collection", "interval_statistics")
else:
    print(f"Config file not found: {config_path}")
    MONGO_URI = "mongodb://localhost:27017/"
    MONGO_DB_NAME = "evo_statistics"
    MONGO_COLLECTION_NAME = "interval_statistics"

WORKSPACE_ID = "01c54ab3-0b97-4b36-8e72-686e65a906ed"  
# OBJECT_ID = "ea6ced6a-31d2-44a9-b8ae-1cee0b5ea42e"     # maia downhole intervals in _ultraenhance
OBJECT_ID = "0286ea01-1a2c-41a8-81b8-fca7df38feac"     # maia downhole collection in _ultraenhance

# Optional: specific version (leave empty for latest)
VERSION = ""

print(f"MongoDB: {MONGO_URI.split('@')[-1] if '@' in MONGO_URI else MONGO_URI}")
print(f"Database: {MONGO_DB_NAME}.{MONGO_COLLECTION_NAME}")
print(f"Evo Object: {OBJECT_ID}")

Using MongoDB Atlas (mongodb+srv): sq-labs-clickops-david.9lxzxzh.mongodb.net
MongoDB: sq-labs-clickops-david.9lxzxzh.mongodb.net/labs-api?retryWrites=true&w=majority&appName=sq-labs-clickops-david
Database: evo_statistics.interval_statistics
Evo Object: 0286ea01-1a2c-41a8-81b8-fca7df38feac


## 3. Connect to MongoDB

Establish connection to MongoDB and create/access the target collection.

Make sure you've built the docker image for the server and that is running before executing this cell.The docker command to run the server is:

```bash
docker run -d -p 27017:27017 --name mongodb mongo:latest
```

In [14]:
def connect_to_mongodb(uri: str, db_name: str, collection_name: str):
    """Connect to MongoDB and return the collection handle."""
    try:
        client = MongoClient(uri, serverSelectionTimeoutMS=5000)
        # Verify connection
        client.admin.command('ping')
        db = client[db_name]
        collection = db[collection_name]
        
        print(f"✓ Connected to MongoDB: {db_name}.{collection_name}")
        return client, db, collection
    except ConnectionFailure as e:
        print(f"✗ Failed to connect to MongoDB: {e}")
        raise

mongo_client, mongo_db, stats_collection = connect_to_mongodb(MONGO_URI, MONGO_DB_NAME, MONGO_COLLECTION_NAME)

✓ Connected to MongoDB: evo_statistics.interval_statistics


## 4. Initialize Evo Connection

Initialize the Evo SDK context and authenticate via OAuth.

In [5]:
# Initialize Evo SDK connection (will trigger OAuth flow if needed)
await ensure_initialized()
print("✓ Evo SDK initialized and authenticated")

✓ Evo SDK initialized and authenticated


## 5. Load Object and Inspect Collections

Download the Evo object and inspect available collections/attributes.

In [15]:
# Download the object
obj, obj_dict = await get_downhole_collection(WORKSPACE_ID, OBJECT_ID, VERSION)

object_name = obj_dict.get('name', 'Unknown')
object_type = get_object_type(obj_dict)

print(f"Object: {object_name}")
print(f"Type: {object_type}")

# Auto-discover all interval tables and their attributes
collections_info = get_collection_info(obj_dict)
total_attributes = 0

print(f"\nDiscovered {len(collections_info)} interval table(s):")
for coll in collections_info:
    attrs = coll.get('attributes', [])
    total_attributes += len(attrs)
    print(f"\n  {coll['name']} ({len(attrs)} attributes)")
    for attr in attrs:
        print(f"     - {attr['name']} ({attr['type']})")

print(f"\n{'='*50}")
print(f"Total: {len(collections_info)} tables × {total_attributes} attributes")
print(f"Will generate {len(collections_info)} MongoDB document(s) (one per interval table)")

Object: Maia Drillholes
Type: downhole-collection

Discovered 2 interval table(s):

  assay (1 attributes)
     - Au (continuous)

  geology (1 attributes)
     - Lithology (continuous)

Total: 2 tables × 2 attributes
Will generate 2 MongoDB document(s) (one per interval table)


## 6. Download All Interval Tables

Download interval data from every discovered collection. For each table, identify
which columns are numeric (suitable for statistics) vs categorical.

In [16]:
# Download all interval tables and classify their columns
import time

collection_data = {}  # dict of { collection_name: { "df": DataFrame, "numeric_cols": [...], "categorical_cols": [...] } }

for coll in collections_info:
    coll_name = coll['name']
    print(f"\n{'='*50}")
    print(f"Downloading: {coll_name}")
    
    t0 = time.perf_counter()
    
    if object_type == 'downhole-intervals':
        df = await download_downhole_intervals_data(obj)
    else:
        df = await download_interval_data(obj, coll_name)
    
    elapsed = time.perf_counter() - t0
    
    # Classify columns: numeric vs categorical
    # Exclude structural columns (hole_id, from, to) from analysis
    structural_cols = {'hole_id', 'from', 'to'}
    attribute_cols = [c for c in df.columns if c not in structural_cols]
    
    numeric_cols = [c for c in attribute_cols if pd.api.types.is_numeric_dtype(df[c])]
    categorical_cols = [c for c in attribute_cols if c not in numeric_cols]
    
    collection_data[coll_name] = {
        "df": df,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
    }
    
    print(f"  ✓ {len(df):,} intervals from {df['hole_id'].nunique():,} holes ({elapsed:.1f}s)")
    print(f"  Numeric attributes ({len(numeric_cols)}): {numeric_cols[:10]}{'...' if len(numeric_cols) > 10 else ''}")
    print(f"  Categorical attributes ({len(categorical_cols)}): {categorical_cols[:10]}{'...' if len(categorical_cols) > 10 else ''}")

print(f"\n{'='*50}")
print(f"Downloaded {len(collection_data)} interval table(s)")


Downloading: assay
  ✓ 2,210 intervals from 14 holes (11.5s)
  Numeric attributes (1): ['Au']
  Categorical attributes (0): []

Downloading: geology
  ✓ 93 intervals from 14 holes (11.4s)
  Numeric attributes (1): ['Lithology']
  Categorical attributes (0): []

Downloaded 2 interval table(s)


## 7. Calculate Statistics for All Attributes

For every numeric attribute on every interval table, compute:
- **Overall**: length-weighted mean, accumulation, min/max/std, count
- **Per-hole**: same metrics grouped by hole_id
- **Gaps**: missing interval analysis per collection

This can produce a large volume of data (e.g., Thalanga assays has ~100 attributes × 5,000 holes),
so we store **one MongoDB document per interval table** to keep documents manageable.

In [17]:
# Calculate statistics for every numeric attribute on every interval table
all_collection_stats = {}  # { collection_name: { "attributes": {...}, "gap_analysis": {...} } }

for coll_name, coll_data in collection_data.items():
    df = coll_data["df"]
    numeric_cols = coll_data["numeric_cols"]
    
    print(f"\n{'='*60}")
    print(f"📊 {coll_name}: {len(numeric_cols)} numeric attributes, {len(df):,} intervals")
    print(f"{'='*60}")
    
    t0 = time.perf_counter()
    attribute_stats = {}
    
    for grade_col in numeric_cols:
        # Overall statistics
        try:
            overall = calculate_interval_statistics(df, grade_col)
        except (ValueError, ZeroDivisionError):
            print(f"{grade_col} skipped (no valid numeric data)")
            continue
        
        # Per-hole statistics
        hole_stats_df = calculate_statistics_by_hole(df, grade_col)
        
        attribute_stats[grade_col] = {
            "overall": overall,
            "by_hole": hole_stats_df.to_dict(orient='records'),
            "hole_count": len(hole_stats_df),
        }
        
        lwm = overall.get('length_weighted_mean', 0)
        count = overall.get('count', 0)
        print(f"{grade_col}: LWM={lwm:.4f}, n={count}, holes={len(hole_stats_df)}")
    
    # Gap analysis (once per collection, not per attribute)
    gap_analysis = analyze_gaps(df)
    
    elapsed = time.perf_counter() - t0
    
    all_collection_stats[coll_name] = {
        "attributes": attribute_stats,
        "gap_analysis": gap_analysis,
    }
    
    print(f"\n  Gaps: {gap_analysis['total_gap_count']} total, {gap_analysis['holes_with_gaps']} holes affected")
    print(f"  Completed in {elapsed:.1f}s ({len(attribute_stats)} attributes analysed)")

print(f"\n{'='*60}")
print(f"Summary: {sum(len(v['attributes']) for v in all_collection_stats.values())} attribute statistics across {len(all_collection_stats)} table(s)")


📊 assay: 1 numeric attributes, 2,210 intervals
Au: LWM=0.4725, n=2210, holes=14

  Gaps: 0 total, 0 holes affected
  Completed in 0.1s (1 attributes analysed)

📊 geology: 1 numeric attributes, 93 intervals
Lithology: LWM=2.8053, n=93, holes=14

  Gaps: 0 total, 0 holes affected
  Completed in 0.0s (1 attributes analysed)

Summary: 2 attribute statistics across 2 table(s)


## 8. Prepare Documents for MongoDB

### Document Strategy

Each object may have multiple interval tables (e.g., `assay`, `geology`), and each table
may have many attributes. Storing everything in one document risks hitting MongoDB's **16 MB
document limit** — especially when per-hole statistics for 100+ attributes across 5,000+ holes
are included.

**Strategy: One document per interval table**, structured as:

```
{
  workspace_id, object_id, object_name, object_type,
  collection_name: "assay",
  stats_summary: [ {grade, lwm, accumulation, max, ...}, ... ],  // flat array for indexing
  grade_statistics: { "Au": {overall: {...}, by_hole: [...]}, ... },
  gap_analysis: {...},
  metadata: { version, timestamp, doc_size_bytes }
}
```

For extremely large tables (estimated doc > 14 MB), we split further into:
- **Summary document**: overall stats + stats_summary array (small, queryable)
- **Detail documents**: per-hole stats chunked by attribute batches

This keeps the summary docs fast to query while preserving full detail.

In [ ]:
# MongoDB document size limit
MONGO_DOC_LIMIT = 16 * 1024 * 1024  # 16 MB
SAFE_DOC_LIMIT = 14 * 1024 * 1024   # 14 MB — leave headroom


def estimate_doc_size(doc: dict) -> int:
    """Estimate BSON document size in bytes."""
    return len(bson.BSON.encode(doc))


def build_stats_summary(attribute_stats: dict) -> list[dict]:
    """Build flat stats_summary array for efficient MongoDB indexing."""
    summary = []
    for grade_name, grade_data in attribute_stats.items():
        overall = grade_data.get("overall", {})
        summary.append({
            "grade": grade_name,
            "lwm": overall.get("length_weighted_mean"),
            "accumulation": overall.get("accumulation_grade_meters"),
            "total_length": overall.get("total_length"),
            "count": overall.get("count"),
            "min": overall.get("min"),
            "max": overall.get("max"),
            "mean": overall.get("mean"),
            "std": overall.get("std"),
            "null_count": overall.get("null_count"),
        })
    return summary


def build_gap_summary(gap_analysis: dict) -> dict:
    """Clean gap analysis for MongoDB (remove DataFrames)."""
    return {
        "total_gap_count": gap_analysis.get("total_gap_count", 0),
        "total_gap_length": gap_analysis.get("total_gap_length", 0),
        "holes_with_gaps": gap_analysis.get("holes_with_gaps", 0),
        "holes_without_gaps": gap_analysis.get("holes_without_gaps", 0),
    }


def prepare_collection_documents(
    workspace_id: str,
    object_id: str,
    object_name: str,
    object_type: str,
    collection_name: str,
    attribute_stats: dict,
    gap_analysis: dict,
) -> list[dict]:
    """Prepare MongoDB documents for one interval table.
    
    Returns a list of documents. Usually 1 document, but if the full document
    would exceed the safe size limit, it splits into:
    - 1 summary doc (overall stats only, small and fast to query)
    - N detail docs (per-hole stats, chunked by attribute batches)
    """
    now = datetime.now(timezone.utc)
    
    base_metadata = {
        "workspace_id": workspace_id,
        "object_id": object_id,
        "object_name": object_name,
        "object_type": object_type,
        "collection_name": collection_name,
        "timestamp": now,
        "metadata": {
            "version": VERSION or "latest",
            "generated_by": "mcp_stats_to_mongo.ipynb",
        },
    }
    
    # Try building a single complete document first
    full_doc = {
        **base_metadata,
        "doc_type": "complete",               # complete | summary | detail
        "stats_summary": build_stats_summary(attribute_stats),
        "grade_statistics": attribute_stats,
        "gap_analysis": build_gap_summary(gap_analysis),
    }
    
    doc_size = estimate_doc_size(full_doc)
    full_doc["metadata"]["doc_size_bytes"] = doc_size
    
    if doc_size < SAFE_DOC_LIMIT:
        return [full_doc]
    
    # --- Document is too large: split into summary + detail chunks ---
    print(f"    {collection_name}: {doc_size / 1024 / 1024:.1f} MB exceeds limit, splitting...")
    
    # Summary document: overall stats only (no per-hole data)
    overall_only = {}
    for grade_name, grade_data in attribute_stats.items():
        overall_only[grade_name] = {
            "overall": grade_data["overall"],
            "hole_count": grade_data.get("hole_count", 0),
            # omit by_hole to keep it small
        }
    
    summary_doc = {
        **base_metadata,
        "doc_type": "summary",
        "stats_summary": build_stats_summary(attribute_stats),
        "grade_statistics": overall_only,
        "gap_analysis": build_gap_summary(gap_analysis),
    }
    summary_doc["metadata"]["doc_size_bytes"] = estimate_doc_size(summary_doc)
    
    documents = [summary_doc]
    
    # Detail documents: chunk per-hole stats by attribute batches
    attr_names = list(attribute_stats.keys())
    chunk_start = 0
    chunk_idx = 0
    
    while chunk_start < len(attr_names):
        # Grow the chunk until it approaches the size limit
        chunk_end = chunk_start + 1
        
        while chunk_end <= len(attr_names):
            chunk_attrs = {
                name: {"by_hole": attribute_stats[name]["by_hole"], "hole_count": attribute_stats[name]["hole_count"]}
                for name in attr_names[chunk_start:chunk_end]
            }
            detail_doc = {
                **base_metadata,
                "doc_type": "detail",
                "chunk_index": chunk_idx,
                "attributes_in_chunk": attr_names[chunk_start:chunk_end],
                "hole_statistics": chunk_attrs,
            }
            if estimate_doc_size(detail_doc) > SAFE_DOC_LIMIT:
                chunk_end -= 1
                break
            chunk_end += 1
        
        # Ensure we make progress (at least 1 attribute per chunk)
        chunk_end = max(chunk_end, chunk_start + 1)
        
        chunk_attrs = {
            name: {"by_hole": attribute_stats[name]["by_hole"], "hole_count": attribute_stats[name]["hole_count"]}
            for name in attr_names[chunk_start:chunk_end]
        }
        detail_doc = {
            **base_metadata,
            "doc_type": "detail",
            "chunk_index": chunk_idx,
            "attributes_in_chunk": attr_names[chunk_start:chunk_end],
            "hole_statistics": chunk_attrs,
        }
        detail_doc["metadata"]["doc_size_bytes"] = estimate_doc_size(detail_doc)
        documents.append(detail_doc)
        
        chunk_start = chunk_end
        chunk_idx += 1
    
    print(f"  Split into 1 summary + {chunk_idx} detail document(s)")
    return documents


# Build all documents
all_documents = []

for coll_name, coll_stats in all_collection_stats.items():
    docs = prepare_collection_documents(
        workspace_id=WORKSPACE_ID,
        object_id=OBJECT_ID,
        object_name=object_name,
        object_type=object_type,
        collection_name=coll_name,
        attribute_stats=coll_stats["attributes"],
        gap_analysis=coll_stats["gap_analysis"],
    )
    all_documents.extend(docs)
    
    for doc in docs:
        size_mb = doc["metadata"].get("doc_size_bytes", 0) / 1024 / 1024
        n_attrs = len(doc.get("stats_summary", doc.get("attributes_in_chunk", [])))
        print(f"  📄 {coll_name} [{doc['doc_type']}]: {size_mb:.2f} MB, {n_attrs} attributes")

print(f"\nTotal documents to insert: {len(all_documents)}")
print(f"Total size: {sum(d['metadata'].get('doc_size_bytes', 0) for d in all_documents) / 1024 / 1024:.2f} MB")

  📄 assay [complete]: 0.00 MB, 1 attributes
  📄 geology [complete]: 0.00 MB, 1 attributes

Total documents to insert: 2
Total size: 0.01 MB


## 9. Insert Statistics into MongoDB

Write the statistics document to the MongoDB collection.

In [20]:
# Insert all documents into MongoDB
if all_documents:
    result = stats_collection.insert_many(all_documents)
    print(f"Inserted {len(result.inserted_ids)} document(s)")
    for i, doc_id in enumerate(result.inserted_ids):
        doc = all_documents[i]
        print(f"  {doc['collection_name']} [{doc['doc_type']}]: {doc_id}")
else:
    print("No documents to insert")

BulkWriteError: batch op errors occurred, full error: {'writeErrors': [{'index': 0, 'code': 11000, 'errmsg': "E11000 duplicate key error collection: evo_statistics.interval_statistics index: _id_ dup key: { _id: ObjectId('698e71c9ef13240590cd7e32') }", 'keyPattern': {'_id': 1}, 'keyValue': {'_id': ObjectId('698e71c9ef13240590cd7e32')}, 'op': {'workspace_id': '01c54ab3-0b97-4b36-8e72-686e65a906ed', 'object_id': '0286ea01-1a2c-41a8-81b8-fca7df38feac', 'object_name': 'Maia Drillholes', 'object_type': 'downhole-collection', 'collection_name': 'assay', 'timestamp': datetime.datetime(2026, 2, 13, 0, 34, 58, 168076, tzinfo=datetime.timezone.utc), 'metadata': {'version': 'latest', 'generated_by': 'mcp_stats_to_mongo.ipynb', 'doc_size_bytes': 3413}, 'doc_type': 'complete', 'stats_summary': [{'grade': 'Au', 'lwm': 0.4725068505663135, 'accumulation': 2586.5025, 'total_length': 5474.0, 'count': 2210, 'min': 0.01, 'max': 5.91, 'mean': None, 'std': 0.6388622250164636, 'null_count': 0}], 'grade_stati

## 10. Create Grade Value Indexes

Create indexes to support queries like "find all high-grade objects" efficiently.

In [21]:
# =============================================================================
# Index for querying by grade values (e.g., "find all high-grade objects")
# Uses array format in stats_summary for flexible querying with a single index
# =============================================================================

def create_grade_stats_indexes(collection):
    """Create indexes optimized for grade value queries."""
    indexes_created = []
    
    # Primary index: query by grade name and length-weighted mean
    # Supports: "find all objects where Au LWM > 1.0"
    collection.create_index(
        [("stats_summary.grade", 1), ("stats_summary.lwm", -1)],
        name="grade_lwm"
    )
    indexes_created.append("grade_lwm")
    
    # Secondary index: query by accumulation (grade-meters)
    # Supports: "find objects with highest Au accumulation"
    collection.create_index(
        [("stats_summary.grade", 1), ("stats_summary.accumulation", -1)],
        name="grade_accumulation"
    )
    indexes_created.append("grade_accumulation")
    
    # Index on max grade value
    # Supports: "find objects with peak Au values > 10"
    collection.create_index(
        [("stats_summary.grade", 1), ("stats_summary.max", -1)],
        name="grade_max"
    )
    indexes_created.append("grade_max")
    
    # Compound index for workspace-scoped grade queries
    # Supports: "find high-grade Au objects in this workspace"
    collection.create_index(
        [("workspace_id", 1), ("stats_summary.grade", 1), ("stats_summary.lwm", -1)],
        name="workspace_grade_lwm"
    )
    indexes_created.append("workspace_grade_lwm")
    
    print(f"✓ Created {len(indexes_created)} grade query indexes: {indexes_created}")
    return indexes_created

# Create the indexes
grade_indexes = create_grade_stats_indexes(stats_collection)

✓ Created 4 grade query indexes: ['grade_lwm', 'grade_accumulation', 'grade_max', 'workspace_grade_lwm']


## 10b. Index Performance Profiling

Compare query performance with and without indexes on the key query patterns.

In [22]:
import time
from statistics import mean, stdev

def profile_query(collection, query: dict, iterations: int = 100) -> dict:
    """Run a query multiple times and measure execution time."""
    times = []
    for _ in range(iterations):
        start = time.perf_counter()
        list(collection.find(query))  # Force cursor evaluation
        end = time.perf_counter()
        times.append((end - start) * 1000)  # Convert to ms
    
    return {
        "mean_ms": round(mean(times), 4),
        "stdev_ms": round(stdev(times), 4) if len(times) > 1 else 0,
        "min_ms": round(min(times), 4),
        "max_ms": round(max(times), 4),
        "iterations": iterations,
    }

def get_explain_stats(collection, query: dict) -> dict:
    """Get query execution stats from explain()."""
    explain = collection.find(query).explain()
    exec_stats = explain.get("executionStats", {})
    return {
        "docs_examined": exec_stats.get("totalDocsExamined", "N/A"),
        "keys_examined": exec_stats.get("totalKeysExamined", "N/A"),
        "execution_time_ms": exec_stats.get("executionTimeMillis", "N/A"),
        "index_used": explain.get("queryPlanner", {}).get("winningPlan", {}).get("inputStage", {}).get("indexName", "COLLSCAN"),
    }

def run_index_benchmark(collection, db):
    """Benchmark queries with and without indexes."""
    
    # Define the two indexes to test
    indexes = [
        {
            "name": "workspace_object_idx",
            "keys": [("workspace_id", 1), ("object_id", 1)],
            "query": {"workspace_id": WORKSPACE_ID, "object_id": OBJECT_ID},
        },
        {
            "name": "object_name_idx",
            "keys": [("object_name", 1)],
            "query": {"object_name": {"$regex": ".*", "$options": "i"}},  # Simulated search
        },
    ]
    
    results = []
    
    for idx_config in indexes:
        print(f"\n{'='*60}")
        print(f"Testing index: {idx_config['name']}")
        print(f"Query: {idx_config['query']}")
        print(f"{'='*60}")
        
        # --- Test WITHOUT index ---
        # Drop the index if it exists
        try:
            collection.drop_index(idx_config["name"])
        except Exception:
            pass  # Index didn't exist
        
        print("\nWITHOUT INDEX:")
        no_idx_profile = profile_query(collection, idx_config["query"])
        no_idx_explain = get_explain_stats(collection, idx_config["query"])
        print(f"   Mean: {no_idx_profile['mean_ms']:.4f} ms (±{no_idx_profile['stdev_ms']:.4f})")
        print(f"   Docs examined: {no_idx_explain['docs_examined']}")
        print(f"   Index used: {no_idx_explain['index_used']}")
        
        # --- Test WITH index ---
        collection.create_index(idx_config["keys"], name=idx_config["name"])
        
        print("\nWITH INDEX:")
        with_idx_profile = profile_query(collection, idx_config["query"])
        with_idx_explain = get_explain_stats(collection, idx_config["query"])
        print(f"   Mean: {with_idx_profile['mean_ms']:.4f} ms (±{with_idx_profile['stdev_ms']:.4f})")
        print(f"   Docs examined: {with_idx_explain['docs_examined']}")
        print(f"   Keys examined: {with_idx_explain['keys_examined']}")
        print(f"   Index used: {with_idx_explain['index_used']}")
        
        # Calculate improvement
        if no_idx_profile['mean_ms'] > 0:
            improvement = ((no_idx_profile['mean_ms'] - with_idx_profile['mean_ms']) / no_idx_profile['mean_ms']) * 100
            print(f"\n   ⚡ Improvement: {improvement:.1f}%")
        
        results.append({
            "index_name": idx_config["name"],
            "without_index": no_idx_profile,
            "with_index": with_idx_profile,
            "docs_examined_without": no_idx_explain['docs_examined'],
            "docs_examined_with": with_idx_explain['docs_examined'],
        })
    
    return results

# Run benchmark — documents are now present in the collection
doc_count = stats_collection.count_documents({})
print(f"Collection has {doc_count} documents")

if doc_count > 0:
    benchmark_results = run_index_benchmark(stats_collection, mongo_db)
else:
    print("No documents found — run the insert cell first")

Collection has 3 documents

Testing index: workspace_object_idx
Query: {'workspace_id': '01c54ab3-0b97-4b36-8e72-686e65a906ed', 'object_id': '0286ea01-1a2c-41a8-81b8-fca7df38feac'}

WITHOUT INDEX:
   Mean: 250.7171 ms (±121.1150)
   Docs examined: 3
   Index used: workspace_grade_lwm

WITH INDEX:
   Mean: 238.6066 ms (±1.6096)
   Docs examined: 2
   Keys examined: 2
   Index used: workspace_object_idx

   ⚡ Improvement: 4.8%

Testing index: object_name_idx
Query: {'object_name': {'$regex': '.*', '$options': 'i'}}

WITHOUT INDEX:
   Mean: 261.1484 ms (±113.7821)
   Docs examined: 3
   Index used: COLLSCAN

WITH INDEX:
   Mean: 238.5091 ms (±3.0952)
   Docs examined: 3
   Keys examined: 3
   Index used: object_name_idx

   ⚡ Improvement: 8.7%


## 10. Verify and Query MongoDB

Query the collection to verify the data was stored correctly and explore historical statistics.

In [23]:
# Collection overview
doc_count = stats_collection.count_documents({})
print(f"Total documents in collection: {doc_count}\n")

# Count by doc_type
for doc_type in ["complete", "summary", "detail"]:
    count = stats_collection.count_documents({"doc_type": doc_type})
    if count > 0:
        print(f"  {doc_type}: {count} document(s)")

# Show documents for this object
print(f"\nDocuments for object {OBJECT_ID}:")
cursor = stats_collection.find(
    {"object_id": OBJECT_ID},
    {"collection_name": 1, "doc_type": 1, "stats_summary": 1, "gap_analysis": 1, "timestamp": 1, "metadata.doc_size_bytes": 1}
).sort([("collection_name", 1), ("doc_type", 1)])

for doc in cursor:
    coll = doc.get("collection_name", "?")
    dtype = doc.get("doc_type", "?")
    size_kb = doc.get("metadata", {}).get("doc_size_bytes", 0) / 1024
    ts = doc.get("timestamp", "")
    
    if dtype in ("complete", "summary"):
        n_attrs = len(doc.get("stats_summary", []))
        gaps = doc.get("gap_analysis", {}).get("total_gap_count", 0)
        print(f"\n  📁 {coll} [{dtype}] — {n_attrs} attributes, {gaps} gaps, {size_kb:.0f} KB")
        
        # Show top 5 attributes by LWM
        summary = sorted(doc.get("stats_summary", []), key=lambda s: abs(s.get("lwm") or 0), reverse=True)
        for s in summary[:5]:
            print(f"     {s['grade']}: LWM={s.get('lwm', 'N/A')}, max={s.get('max', 'N/A')}, n={s.get('count', 'N/A')}")
        if len(summary) > 5:
            print(f"     ... and {len(summary) - 5} more")
    else:
        attrs = doc.get("attributes_in_chunk", [])
        print(f"\n  📁 {coll} [{dtype}] — {len(attrs)} attributes (detail chunk), {size_kb:.0f} KB")

Total documents in collection: 3

  complete: 3 document(s)

Documents for object 0286ea01-1a2c-41a8-81b8-fca7df38feac:

  📁 assay [complete] — 1 attributes, 0 gaps, 3 KB
     Au: LWM=0.4725068505663135, max=5.91, n=2210

  📁 geology [complete] — 1 attributes, 0 gaps, 3 KB
     Lithology: LWM=2.8052795031055897, max=5.0, n=93


## 11. Query High-Grade Objects

Example queries for finding objects by grade statistics.

In [ ]:
# =============================================================================
# High-Grade Object Queries
# =============================================================================

def find_high_grade_objects(
    collection,
    grade: str,
    min_lwm: float = None,
    min_max: float = None,
    min_accumulation: float = None,
    workspace_id: str = None,
    limit: int = 20,
) -> list[dict]:
    """
    Find objects with high grade values.
    
    Args:
        grade: Grade column name (e.g., "Au", "Cu")
        min_lwm: Minimum length-weighted mean
        min_max: Minimum peak/max value
        min_accumulation: Minimum accumulation (grade-meters)
        workspace_id: Optional workspace filter
        limit: Max results to return
    """
    # Build the $elemMatch query for the stats_summary array
    elem_match = {"grade": grade}
    
    if min_lwm is not None:
        elem_match["lwm"] = {"$gte": min_lwm}
    if min_max is not None:
        elem_match["max"] = {"$gte": min_max}
    if min_accumulation is not None:
        elem_match["accumulation"] = {"$gte": min_accumulation}
    
    query = {"stats_summary": {"$elemMatch": elem_match}}
    
    if workspace_id:
        query["workspace_id"] = workspace_id
    
    # Sort by LWM descending
    results = list(collection.find(
        query,
        {"object_id": 1, "object_name": 1, "stats_summary": 1, "timestamp": 1}
    ).sort([("stats_summary.lwm", -1)]).limit(limit))
    
    return results


def get_top_objects_by_grade(collection, grade: str, metric: str = "lwm", top_n: int = 10) -> list[dict]:
    """
    Get top N objects ranked by a grade metric.
    
    Args:
        grade: Grade column name
        metric: One of "lwm", "max", "accumulation"
        top_n: Number of results
    """
    pipeline = [
        {"$unwind": "$stats_summary"},
        {"$match": {"stats_summary.grade": grade}},
        {"$sort": {f"stats_summary.{metric}": -1}},
        {"$limit": top_n},
        {"$project": {
            "object_id": 1,
            "object_name": 1,
            "workspace_id": 1,
            "grade": "$stats_summary.grade",
            metric: f"$stats_summary.{metric}",
            "timestamp": 1,
        }}
    ]
    return list(collection.aggregate(pipeline))


# Example queries - uncomment to run:

# 1. Find all objects with Au length-weighted mean >= 1.0 g/t
high_au_objects = find_high_grade_objects(stats_collection, grade="Au", min_lwm=1.0)
print(f"Found {len(high_au_objects)} objects with Au LWM >= 1.0")

# 2. Find objects with Au peak values >= 10 g/t
# high_peak_objects = find_high_grade_objects(stats_collection, grade="Au", min_max=10.0)

# 3. Top 10 objects by Au length-weighted mean
# top_au = get_top_objects_by_grade(stats_collection, grade="Au", metric="lwm", top_n=10)
# for obj in top_au:
#     print(f"  {obj['object_name']}: {obj['lwm']:.4f}")

# 4. Find high-grade objects in a specific workspace
# workspace_high_grade = find_high_grade_objects(
#     stats_collection, 
#     grade="Au", 
#     min_lwm=0.5, 
#     workspace_id=WORKSPACE_ID
# )

# print("High-grade query functions defined. Uncomment examples above to run.")

Found 0 objects with Au LWM >= 1.0
High-grade query functions defined. Uncomment examples above to run.


## 11b. Profile High-Grade Query Performance

Benchmark the high-grade query functions with and without indexes.

In [26]:
import time
from statistics import mean, stdev

def profile_high_grade_queries(collection, grade: str = "Au", iterations: int = 50):
    """
    Profile high-grade query performance with and without indexes.
    
    Tests:
    1. find_high_grade_objects with $elemMatch
    2. get_top_objects_by_grade aggregation pipeline
    """
    
    # Define the indexes we're testing
    grade_indexes = [
        ("grade_lwm", [("stats_summary.grade", 1), ("stats_summary.lwm", -1)]),
        ("grade_max", [("stats_summary.grade", 1), ("stats_summary.max", -1)]),
        ("grade_accumulation", [("stats_summary.grade", 1), ("stats_summary.accumulation", -1)]),
        ("workspace_grade_lwm", [("workspace_id", 1), ("stats_summary.grade", 1), ("stats_summary.lwm", -1)]),
    ]
    
    # Define test queries
    test_cases = [
        {
            "name": f"find_high_grade_objects({grade}, min_lwm=0.5)",
            "query": {"stats_summary": {"$elemMatch": {"grade": grade, "lwm": {"$gte": 0.5}}}},
            "type": "find",
        },
        {
            "name": f"find_high_grade_objects({grade}, min_max=5.0)",
            "query": {"stats_summary": {"$elemMatch": {"grade": grade, "max": {"$gte": 5.0}}}},
            "type": "find",
        },
        {
            "name": f"get_top_objects_by_grade({grade}, lwm, top_n=10)",
            "pipeline": [
                {"$unwind": "$stats_summary"},
                {"$match": {"stats_summary.grade": grade}},
                {"$sort": {"stats_summary.lwm": -1}},
                {"$limit": 10},
            ],
            "type": "aggregate",
        },
    ]
    
    results = []
    
    # --- Test WITHOUT indexes ---
    print("=" * 70)
    print("DROPPING GRADE INDEXES FOR BASELINE...")
    print("=" * 70)
    
    for idx_name, _ in grade_indexes:
        try:
            collection.drop_index(idx_name)
        except Exception:
            pass
    
    print("\n📊 WITHOUT INDEXES:\n")
    no_idx_results = {}
    
    for test in test_cases:
        times = []
        for _ in range(iterations):
            start = time.perf_counter()
            if test["type"] == "find":
                list(collection.find(test["query"]).limit(20))
            else:
                list(collection.aggregate(test["pipeline"]))
            end = time.perf_counter()
            times.append((end - start) * 1000)
        
        stats = {
            "mean_ms": round(mean(times), 4),
            "stdev_ms": round(stdev(times), 4) if len(times) > 1 else 0,
            "min_ms": round(min(times), 4),
            "max_ms": round(max(times), 4),
        }
        no_idx_results[test["name"]] = stats
        
        # Get explain for find queries
        if test["type"] == "find":
            explain = collection.find(test["query"]).explain()
            exec_stats = explain.get("executionStats", {})
            docs_examined = exec_stats.get("totalDocsExamined", "N/A")
            index_used = explain.get("queryPlanner", {}).get("winningPlan", {}).get("inputStage", {}).get("indexName", "COLLSCAN")
        else:
            docs_examined = "N/A (aggregate)"
            index_used = "N/A (aggregate)"
        
        print(f"   {test['name']}")
        print(f"      Mean: {stats['mean_ms']:.4f} ms (±{stats['stdev_ms']:.4f})")
        print(f"      Docs examined: {docs_examined}, Index: {index_used}\n")
    
    # --- Test WITH indexes ---
    print("\n" + "=" * 70)
    print("CREATING GRADE INDEXES...")
    print("=" * 70)
    
    for idx_name, idx_keys in grade_indexes:
        collection.create_index(idx_keys, name=idx_name)
    print(f"Created: {[name for name, _ in grade_indexes]}")
    
    print("\n📊 WITH INDEXES:\n")
    with_idx_results = {}
    
    for test in test_cases:
        times = []
        for _ in range(iterations):
            start = time.perf_counter()
            if test["type"] == "find":
                list(collection.find(test["query"]).limit(20))
            else:
                list(collection.aggregate(test["pipeline"]))
            end = time.perf_counter()
            times.append((end - start) * 1000)
        
        stats = {
            "mean_ms": round(mean(times), 4),
            "stdev_ms": round(stdev(times), 4) if len(times) > 1 else 0,
            "min_ms": round(min(times), 4),
            "max_ms": round(max(times), 4),
        }
        with_idx_results[test["name"]] = stats
        
        # Get explain for find queries
        if test["type"] == "find":
            explain = collection.find(test["query"]).explain()
            exec_stats = explain.get("executionStats", {})
            docs_examined = exec_stats.get("totalDocsExamined", "N/A")
            keys_examined = exec_stats.get("totalKeysExamined", "N/A")
            index_used = explain.get("queryPlanner", {}).get("winningPlan", {}).get("inputStage", {}).get("indexName", "COLLSCAN")
        else:
            docs_examined = "N/A"
            keys_examined = "N/A"
            index_used = "N/A (aggregate)"
        
        print(f"   {test['name']}")
        print(f"      Mean: {stats['mean_ms']:.4f} ms (±{stats['stdev_ms']:.4f})")
        print(f"      Docs examined: {docs_examined}, Keys: {keys_examined}, Index: {index_used}\n")
    
    # --- Summary ---
    print("\n" + "=" * 70)
    print("PERFORMANCE COMPARISON SUMMARY")
    print("=" * 70)
    print(f"{'Query':<50} {'No Index':>12} {'With Index':>12} {'Improvement':>12}")
    print("-" * 86)
    
    for test_name in no_idx_results:
        no_idx = no_idx_results[test_name]["mean_ms"]
        with_idx = with_idx_results[test_name]["mean_ms"]
        if no_idx > 0:
            improvement = ((no_idx - with_idx) / no_idx) * 100
            improvement_str = f"{improvement:+.1f}%"
        else:
            improvement_str = "N/A"
        
        # Truncate long names
        display_name = test_name[:48] + ".." if len(test_name) > 50 else test_name
        print(f"{display_name:<50} {no_idx:>10.4f}ms {with_idx:>10.4f}ms {improvement_str:>12}")
        
        results.append({
            "query": test_name,
            "no_index_ms": no_idx,
            "with_index_ms": with_idx,
            "improvement_pct": improvement if no_idx > 0 else None,
        })
    
    return results


# Run the profiling
doc_count = stats_collection.count_documents({})
print(f"Collection has {doc_count} documents\n")

if doc_count > 0:
    # Use first available grade column or default to "Au"
    test_grade = available_grade_columns[0] if 'available_grade_columns' in dir() and available_grade_columns else "Au"
    print(f"Profiling with grade: {test_grade}\n")
    grade_query_results = profile_high_grade_queries(stats_collection, grade=test_grade)
else:
    print("⚠ Insert documents first, then run this cell for profiling")

Collection has 3 documents

Profiling with grade: Au

DROPPING GRADE INDEXES FOR BASELINE...

📊 WITHOUT INDEXES:

   find_high_grade_objects(Au, min_lwm=0.5)
      Mean: 252.2263 ms (±2.5760)
      Docs examined: 3, Index: COLLSCAN

   find_high_grade_objects(Au, min_max=5.0)
      Mean: 252.6385 ms (±2.6154)
      Docs examined: 3, Index: COLLSCAN

   get_top_objects_by_grade(Au, lwm, top_n=10)
      Mean: 247.0964 ms (±7.7675)
      Docs examined: N/A (aggregate), Index: N/A (aggregate)


CREATING GRADE INDEXES...
Created: ['grade_lwm', 'grade_max', 'grade_accumulation', 'workspace_grade_lwm']

📊 WITH INDEXES:

   find_high_grade_objects(Au, min_lwm=0.5)
      Mean: 251.9890 ms (±1.5637)
      Docs examined: 0, Keys: 0, Index: grade_lwm

   find_high_grade_objects(Au, min_max=5.0)
      Mean: 251.8856 ms (±0.7771)
      Docs examined: 1, Keys: 1, Index: grade_max

   get_top_objects_by_grade(Au, lwm, top_n=10)
      Mean: 252.5094 ms (±2.4498)
      Docs examined: N/A, Keys: N/A, Ind

## 12. MCP Tool for Grade Statistics Queries

Define an MCP tool that can be registered with FastMCP to query the MongoDB statistics database via natural language.

In [ ]:
# =============================================================================
# MCP Tool for MongoDB Grade Statistics Queries
# This can be copied to src/evo_mcp/tools/mongo_stats_tools.py
# =============================================================================

from typing import Optional, Literal
from pymongo import MongoClient

# MongoDB connection (would be configured via environment in production)
def get_stats_collection():
    """Get the MongoDB stats collection. Configure via env vars in production."""
    client = MongoClient(MONGO_URI)
    return client[MONGO_DB_NAME][MONGO_COLLECTION_NAME]


def register_mongo_stats_tools(mcp):
    """Register MongoDB statistics query tools with the FastMCP server."""
    
    @mcp.tool()
    async def query_grade_statistics(
        grade: str,
        query_type: Literal["find_above_threshold", "top_n", "compare"],
        metric: Literal["lwm", "max", "accumulation"] = "lwm",
        threshold: Optional[float] = None,
        top_n: int = 10,
        workspace_id: Optional[str] = None,
    ) -> dict:
        """Query the MongoDB statistics database to find objects by their grade values.
        
        Use this tool when users ask about:
        - Finding high-grade objects (e.g., "find all high gold deposits")
        - Comparing grades across objects (e.g., "which objects have Au > 1 g/t")
        - Ranking objects by grade (e.g., "top 10 copper-rich objects")
        - Filtering by thresholds (e.g., "objects with peak silver above 50 g/t")
        
        Args:
            grade: Element/grade column name (Au, Cu, Ag, Pb, Zn, etc.)
            query_type: Type of query - find_above_threshold, top_n, or compare
            metric: Which metric to query - lwm (length-weighted mean), max (peak), accumulation
            threshold: Minimum value threshold (required for find_above_threshold)
            top_n: Number of results for top_n queries (default 10)
            workspace_id: Optional workspace UUID to scope the search
            
        Returns:
            Dict with query_description, count, and results array
        """
        collection = get_stats_collection()
        
        metric_names = {
            "lwm": "length-weighted mean",
            "max": "peak value",
            "accumulation": "accumulation (grade-meters)",
        }
        metric_desc = metric_names.get(metric, metric)
        
        if query_type == "find_above_threshold":
            if threshold is None:
                return {"status": "error", "error": "threshold is required for find_above_threshold queries"}
            
            elem_match = {"grade": grade, metric: {"$gte": threshold}}
            query = {"stats_summary": {"$elemMatch": elem_match}}
            if workspace_id:
                query["workspace_id"] = workspace_id
            
            results = list(collection.find(
                query,
                {"object_id": 1, "object_name": 1, "workspace_id": 1, "stats_summary": 1, "_id": 0}
            ).sort([(f"stats_summary.{metric}", -1)]).limit(50))
            
            # Extract just the relevant grade from stats_summary
            for r in results:
                r["grade_stats"] = next(
                    (s for s in r.get("stats_summary", []) if s["grade"] == grade), 
                    None
                )
                if "stats_summary" in r:
                    del r["stats_summary"]
            
            return {
                "status": "success",
                "query_description": f"Objects with {grade} {metric_desc} ≥ {threshold}",
                "count": len(results),
                "results": results,
            }
        
        elif query_type == "top_n":
            pipeline = [
                {"$unwind": "$stats_summary"},
                {"$match": {"stats_summary.grade": grade}},
            ]
            if workspace_id:
                pipeline.insert(0, {"$match": {"workspace_id": workspace_id}})
            
            pipeline.extend([
                {"$sort": {f"stats_summary.{metric}": -1}},
                {"$limit": top_n},
                {"$project": {
                    "_id": 0,
                    "object_id": 1,
                    "object_name": 1,
                    "workspace_id": 1,
                    "grade": "$stats_summary.grade",
                    metric: f"$stats_summary.{metric}",
                }}
            ])
            
            results = list(collection.aggregate(pipeline))
            
            return {
                "status": "success",
                "query_description": f"Top {top_n} objects by {grade} {metric_desc}",
                "count": len(results),
                "results": results,
            }
        
        elif query_type == "compare":
            pipeline = [
                {"$unwind": "$stats_summary"},
                {"$match": {"stats_summary.grade": grade}},
            ]
            if workspace_id:
                pipeline.insert(0, {"$match": {"workspace_id": workspace_id}})
            
            pipeline.extend([
                {"$project": {
                    "_id": 0,
                    "object_id": 1,
                    "object_name": 1,
                    "lwm": "$stats_summary.lwm",
                    "max": "$stats_summary.max",
                    "accumulation": "$stats_summary.accumulation",
                }},
                {"$sort": {metric: -1}},
            ])
            
            results = list(collection.aggregate(pipeline))
            
            return {
                "status": "success",
                "query_description": f"All objects with {grade} statistics for comparison",
                "count": len(results),
                "results": results,
            }
        
        return {"status": "error", "error": f"Unknown query_type: {query_type}"}

    @mcp.tool()
    async def list_available_grades(
        workspace_id: Optional[str] = None,
    ) -> dict:
        """List all grade columns available in the statistics database.
        
        Use this to discover what grades/elements are available before querying.
        
        Args:
            workspace_id: Optional workspace UUID to scope the search
            
        Returns:
            Dict with list of available grades and their object counts
        """
        collection = get_stats_collection()
        
        pipeline = [
            {"$unwind": "$stats_summary"},
        ]
        if workspace_id:
            pipeline.insert(0, {"$match": {"workspace_id": workspace_id}})
        
        pipeline.extend([
            {"$group": {
                "_id": "$stats_summary.grade",
                "object_count": {"$sum": 1},
                "avg_lwm": {"$avg": "$stats_summary.lwm"},
                "max_value": {"$max": "$stats_summary.max"},
            }},
            {"$sort": {"object_count": -1}},
        ])
        
        results = list(collection.aggregate(pipeline))
        
        grades = [
            {
                "grade": r["_id"],
                "object_count": r["object_count"],
                "average_lwm": round(r["avg_lwm"], 4) if r["avg_lwm"] else None,
                "highest_peak": round(r["max_value"], 4) if r["max_value"] else None,
            }
            for r in results
        ]
        
        return {
            "status": "success",
            "grade_count": len(grades),
            "grades": grades,
        }

    @mcp.tool()
    async def get_grade_statistics_summary(
        object_id: str,
    ) -> dict:
        """Get the grade statistics summary for a specific object.
        
        Args:
            object_id: The object UUID to look up
            
        Returns:
            Dict with the object's grade statistics
        """
        collection = get_stats_collection()
        
        result = collection.find_one(
            {"object_id": object_id},
            {"_id": 0, "object_name": 1, "object_type": 1, "stats_summary": 1, "gap_analysis": 1, "timestamp": 1},
            sort=[("timestamp", -1)]
        )
        
        if not result:
            return {"status": "error", "error": f"No statistics found for object {object_id}"}
        
        return {
            "status": "success",
            "object_id": object_id,
            "object_name": result.get("object_name"),
            "object_type": result.get("object_type"),
            "timestamp": result.get("timestamp").isoformat() if result.get("timestamp") else None,
            "stats_summary": result.get("stats_summary", []),
            "gap_analysis": result.get("gap_analysis", {}),
        }


# =============================================================================
# Example: How this would be registered in mcp_tools.py
# =============================================================================

REGISTRATION_EXAMPLE = '''
# In src/mcp_tools.py, add:

from evo_mcp.tools.mongo_stats_tools import register_mongo_stats_tools

# Then in the tool registration section:
if TOOL_FILTER in ["all", "data"]:
    register_mongo_stats_tools(mcp)
'''

print("MCP tools defined:")
print("  - query_grade_statistics: Find objects by grade values")
print("  - list_available_grades: Discover available grade columns")
print("  - get_grade_statistics_summary: Get stats for a specific object")
print("\nTo register with FastMCP, see the registration example above.")

: 

: 

: 

: 

: 

## 12b. Test MCP Tools Locally

Simulate MCP tool calls as an agent would invoke them.

In [ ]:
# =============================================================================
# Test the MCP tools locally (simulating how an agent would call them)
# =============================================================================

# Create a mock MCP server to register and test the tools
from fastmcp import FastMCP

test_mcp = FastMCP("Test Stats Server")
register_mongo_stats_tools(test_mcp)

print("Registered tools:", [t.name for t in test_mcp._tool_manager._tools.values()])

# Test the tools directly
async def test_mcp_tools():
    doc_count = stats_collection.count_documents({})
    if doc_count == 0:
        print("⚠ Insert documents first, then run this cell to test")
        return
    
    print(f"\nTesting against {doc_count} documents:\n")
    print("=" * 70)
    
    # Test 1: List available grades
    print("🔧 Tool: list_available_grades()")
    result = await test_mcp._tool_manager._tools["list_available_grades"].fn()
    print(f"   Status: {result['status']}")
    print(f"   Found {result['grade_count']} grades:")
    for g in result.get('grades', [])[:5]:
        print(f"     - {g['grade']}: {g['object_count']} objects, avg LWM={g.get('average_lwm', 'N/A')}")
    
    print("\n" + "=" * 70)
    
    # Test 2: Query high-grade objects
    print("🔧 Tool: query_grade_statistics(grade='Au', query_type='find_above_threshold', threshold=0.5)")
    result = await test_mcp._tool_manager._tools["query_grade_statistics"].fn(
        grade="Au",
        query_type="find_above_threshold",
        metric="lwm",
        threshold=0.5,
    )
    print(f"   Status: {result['status']}")
    print(f"   {result.get('query_description', '')}")
    print(f"   Found {result.get('count', 0)} results")
    
    print("\n" + "=" * 70)
    
    # Test 3: Top N query
    print("🔧 Tool: query_grade_statistics(grade='Au', query_type='top_n', top_n=5)")
    result = await test_mcp._tool_manager._tools["query_grade_statistics"].fn(
        grade="Au",
        query_type="top_n",
        metric="lwm",
        top_n=5,
    )
    print(f"   Status: {result['status']}")
    print(f"   {result.get('query_description', '')}")
    if result.get('results'):
        print("   Results:")
        for r in result['results']:
            print(f"     - {r.get('object_name', r.get('object_id'))}: {r.get('lwm', 'N/A')}")

# Run the tests
await test_mcp_tools()

: 

: 

: 

: 

: 